# Expense Tracker — Initial ML Model Training (Kaggle)

This notebook is the canonical Kaggle entry point for the **initial ML training run**. It trains every model currently supported by the master pipeline, evaluates the selected category classifier, preserves the leakage-safe test process, and packages the complete run for download.

### Models produced
- TF-IDF category classifier
- XLM-R Transformer category classifier
- Merchant similarity index
- Duplicate similarity model
- Transaction anomaly model (when amount features exist)
- Spending forecast model (when date + amount features exist)

**Run this notebook with a Kaggle GPU accelerator enabled.** The production pipeline uses Python 3.14 through `uv`, so the notebook does not depend on Kaggle's kernel Python version.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
    import sys

REPO_URL = "https://github.com/Yoge-2004/expense-tracker.git"
REPO_REF = "feature/ml-expense-intelligence"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "expense-tracker"
ML_DIR = REPO_DIR / "ml"
RUNS_DIR = WORK_ROOT / "expense_ml_initial_runs"
CACHE_ROOT = Path("/kaggle/temp/huggingface")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_ROOT)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_ROOT / "transformers")

print("Repository:", REPO_URL)
print("Branch:", REPO_REF)
print("Kaggle working directory:", WORK_ROOT)

In [ ]:
def run(*args, cwd=None, env=None, check=True):
    merged_env = os.environ.copy()
    if env:
        merged_env.update({k: str(v) for k, v in env.items()})
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, cwd=cwd, env=merged_env, check=check)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR))
print("Checked out:", REPO_DIR)
run("git", "rev-parse", "HEAD", cwd=REPO_DIR)

## 1. Verify the Kaggle accelerator

In [ ]:
gpu_check = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU')"],
    capture_output=True, text=True
)
print(gpu_check.stdout)
try:
    run("nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader")
except Exception as exc:
    print("nvidia-smi check failed before project environment setup:", exc)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No Kaggle GPU was detected. Enable a GPU accelerator before training the Transformer.")

## 2. Bootstrap Python 3.14 with uv

In [ ]:
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(uv).exists():
    raise FileNotFoundError("uv was installed but could not be located")
run(uv, "python", "install", "3.14")
run(uv, "sync", "--all-extras", "--dev", cwd=ML_DIR)
run(uv, "run", "python", "--version", cwd=ML_DIR)
run(uv, "run", "python", "-c", "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')", cwd=ML_DIR)

## 3. Load the Hugging Face token from Kaggle Secrets

Create a Kaggle Secret named `HF_TOKEN` (or `HUGGINGFACE_TOKEN`) before running this cell. The token is used for gated/private Hugging Face access and is never written to the repository.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = secrets.get_secret(name)
        except Exception:
            token = None
        if token:
            os.environ["HF_TOKEN"] = token
            break
except Exception as exc:
    print("Kaggle Secrets API unavailable:", exc)

if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN was not found. Public datasets can still work, but gated/private sources may fail.")
else:
    print("HF_TOKEN loaded from Kaggle Secrets.")

## 4. Force a completely fresh dataset preparation

The initial run should not reuse prepared data or source-part caches from an older experiment. The cache directories are disposable on Kaggle.

In [ ]:
data_dir = ML_DIR / "data"
for path in (data_dir / "prepared", data_dir / "cache", data_dir / "raw"):
    if path.exists():
        print("Removing:", path)
        shutil.rmtree(path)

for path in (ML_DIR / "artifacts", RUNS_DIR):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

print("Fresh initial-training workspace is ready.")

## 5. Kaggle training resource configuration

The master pipeline will use the Transformer, mixed precision, deterministic sampling, validation model selection, final refit, untouched test evaluation, and auxiliary models. These are Kaggle-specific resource settings; they do not change production defaults in the repository.

In [ ]:
cpu_count = os.cpu_count() or 4
kaggle_env = {
    "EXPENSE_ML_CPU_THREADS": min(cpu_count, 8),
    "EXPENSE_ML_TORCH_THREADS": min(cpu_count, 8),
    "EXPENSE_ML_DATALOADER_WORKERS": min(max(cpu_count // 2, 1), 4),
    "EXPENSE_ML_BATCH_SIZE": 16,
    "EXPENSE_ML_EVAL_BATCH_SIZE": 64,
    "EXPENSE_ML_MIXED_PRECISION": "auto",
    "EXPENSE_ML_NO_PROGRESS": "0",
}
os.environ.update({k: str(v) for k, v in kaggle_env.items()})
print(json.dumps(kaggle_env, indent=2))

## 6. Prepare the configured datasets

In [ ]:
run(uv, "run", "expense-ml", "prepare", "--config", "config/datasets.yaml", "--output", "data/prepared/transactions.parquet", cwd=ML_DIR, env=kaggle_env)
prepared = ML_DIR / "data/prepared/transactions.parquet"
if not prepared.exists():
    raise FileNotFoundError(f"Prepared dataset was not created: {prepared}")
run(uv, "run", "python", "-c", "import pyarrow.parquet as pq; p='data/prepared/transactions.parquet'; print('Prepared rows:', pq.ParquetFile(p).metadata.num_rows)", cwd=ML_DIR)

## 7. Run the complete initial training pipeline

This is the main training cell. Do not use `--no-transformer` here: the initial model should include the Transformer candidate so model selection can compare it against TF-IDF.

In [ ]:
run(
    uv, "run", "python", "-m", "expense_ml.master_pipeline",
    "--config", "config/datasets.yaml",
    "--prepared", "data/prepared/transactions.parquet",
    "--output", str(RUNS_DIR),
    cwd=ML_DIR,
    env=kaggle_env,
)

## 8. Inspect the completed run

In [ ]:
run_dirs = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError("No master-training run directory was produced.")
RUN_DIR = run_dirs[-1]
manifest_path = RUN_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Run:", RUN_DIR.name)
print("Status:", manifest.get("status"))
print("Pipeline version:", manifest.get("pipeline_version"))
print("Selected model:", manifest.get("selected_model"))
print("Test evaluation:", manifest.get("test_evaluation"))
print("India holdout:", manifest.get("india_holdout_evaluation"))

In [ ]:
for relative in (
    "reports/dataset_quality.json",
    "reports/split_summary.json",
    "reports/model_selection_validation.json",
    "reports/model_comparison.json",
    "reports/category_test.json",
    "reports/category_test_country_metrics.json",
):
    path = RUN_DIR / relative
    print(f"\n--- {relative} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:12000])
    else:
        print("MISSING")

## 9. Package the complete initial model run

The archive contains all trained model artifacts and reports. No raw dataset files are intentionally added to the archive.

In [ ]:
archive_base = WORK_ROOT / f"expense-tracker-ml-initial-run-{RUN_DIR.name}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Complete run archive:", archive_path)
print("Archive size (MiB):", round(archive_path.stat().st_size / 1024**2, 2))

## 10. Optional: publish the selected category model to Hugging Face

Leave `PUBLISH_TO_HF = False` for a train-only run. Set it to `True` only after inspecting the metrics. This publishes the selected category model to a versioned revision and advances the `production` revision. The model repository should be private.

In [ ]:
PUBLISH_TO_HF = False
HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "Yoge-2004/expense-intelligence-model")
HF_MODEL_PRIVATE = os.environ.get("HF_MODEL_PRIVATE", "1") == "1"

if PUBLISH_TO_HF:
    token = os.environ.get("HF_TOKEN", "")
    if not token:
        raise RuntimeError("HF_TOKEN is required for model publication.")
    selected = manifest["selected_model"]["name"]
    selected_dir = RUN_DIR / "models" / ("category-transformer" if selected == "transformer" else "category-tfidf")
    if not selected_dir.exists():
        raise FileNotFoundError(f"Selected model directory not found: {selected_dir}")
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE, exist_ok=True)
    revision = f"v{RUN_DIR.name}"
    api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=revision, revision="main", exist_ok=True)
    api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch="production", revision="main", exist_ok=True)
    for target in (revision, "production"):
        api.upload_folder(
            repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(selected_dir),
            revision=target, commit_message=f"Publish Expense Tracker initial model {revision}"
        )
        api.upload_file(
            path_or_fileobj=str(manifest_path), path_in_repo="manifest.json",
            repo_id=HF_MODEL_REPO, repo_type="model", revision=target,
            commit_message="Publish initial training manifest"
        )
    print({"repo_id": HF_MODEL_REPO, "revision": revision, "production": "production"})
else:
    print("Publication skipped. Inspect the metrics and archive first.")

## 11. Final Kaggle outputs

The canonical artifact to keep is the ZIP archive created above. It contains `manifest.json`, all model directories, evaluation reports, and figures. Use the selected model metrics in the notebook output to decide whether the model is ready for the serving/deployment stage.